# Six PLS linear solvers: dense Cholesky, banded Cholesky, and nonzero $\mu$

This is a **separate, reproducible companion study** to the four-method timing experiment in the paper. It adds two *genuine* Cholesky algorithms without relabeling any previous `solve_banded` measurements:

1. Dense normal-equations solve (fresh $Q$ per penalty).
2. Dense normal-equations solve (cached $Q$).
3. Reusable spectral SVD.
4. General banded solver (`scipy.linalg.solve_banded`).
5. **Dense Cholesky** (`scipy.linalg.cho_factor` + `cho_solve`).
6. **Banded Cholesky** (`scipy.linalg.cholesky_banded` + `cho_solve_banded`).

The objective for fixed numeric $\mu$ is
$$\min_{\tau}\left\{\|y-\tau\|_2^2+\lambda\|D_d\tau-\mu\mathbf{1}\|_2^2\right\}.$$

Differentiating yields
$$(I+\lambda D_d^\top D_d)\widehat\tau=y+\lambda\mu D_d^\top\mathbf1.$$

Thus $\mu\ne0$ changes the **right-hand side**, not the matrix. All methods solve the same system for each shared penalty. The original paper's four-method timing tables remain unchanged.

We time setup and a complete sweep of $M$ fitted trends, comparing the same $n,d,M$, seed, and penalties for $\mu=0$ and $\mu=0.15$. We also measure separate trace/EDF workloads, which do not depend on $\mu$.

In [ ]:
import awesomefigures as af
from matplotlib_inline.backend_inline import set_matplotlib_formats
af.set_style("elegante", font="DejaVu Sans", use_tex=False,
             palette="okabe-ito", rasterize=False)
set_matplotlib_formats("svg", "png")
from trendestimation._docs_style import apply_documentation_figure_style
apply_documentation_figure_style()


## Data provenance and six-way parity

This notebook loads **measured CSV files** from `paper/benchmarks/cholesky_reference/`. The committed reference data are generated by a GitHub Actions runner and labeled with its own OS, CPU, libraries, BLAS-thread count, timestamp, and commit.

On your Windows 11 VM, run `python paper/benchmarks/cholesky_comparison.py --preset full --repeats 7` to measure your own hardware. Results default to `paper/benchmarks/results/six_solver/`; optionally pass `--output-dir paper/benchmarks/cholesky_reference` to regenerate the notebook's input files. Do not interpret different hosts' times as directly paired evidence.

In [ ]:
import csv
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/"paper/benchmarks/cholesky_reference/full_fit_timings.csv").is_file()), None)
if root is None:
    raise FileNotFoundError("Expected benchmark reference CSVs. Run "
        "paper/benchmarks/cholesky_comparison.py --output-dir "
        "paper/benchmarks/cholesky_reference from the repository root.")
folder = root/"paper/benchmarks/cholesky_reference"
def load_rows(file):
    with (folder/file).open(encoding="utf-8",newline="") as stream:
        return list(csv.DictReader(stream))
fit_rows=load_rows("full_fit_timings.csv")
trace_rows=load_rows("trace_timings.csv")
env=json.loads((folder/"environment.json").read_text(encoding="utf-8"))
print("Recorded platform:",env["system"])
print("Revision:",env.get("git_revision","unknown"))
print("BLAS threads:",env["blas_threads"],"repetitions:",env["repeats"])
print(len(fit_rows),"fit rows;",len(trace_rows),"trace rows")
names=("dense_fresh","dense_cached","spectral","banded",
       "dense_cholesky","banded_cholesky")
pretty=("Dense fresh","Dense cached","SVD","General banded",
        "Dense Cholesky","Banded Cholesky")
colors=("#80888f","#c98b35","#205fd4","#138b94","#a45cb9","#d0576b")
assert set(r["method"] for r in fit_rows)==set(names)
assert set(r["method"] for r in trace_rows)==set(n+"_trace" for n in names)
assert max(float(r["max_error_banded"]) for r in fit_rows)<3e-6
assert max(float(r["max_abs_trace_error"]) for r in trace_rows)<1e-6


## Six-way complete-fit comparison for both $\mu$ values

The panels are **total median runtime** including setup and all $M$ fits, not just the last linear solve. With $d=2$, we show $n=100$ and both mean assumptions separately. Identical simulated observations and penalty grids are used for paired $\mu$ cases. We do not interpret small differences between $\mu$ conditions as a fundamental change in the factorization.

In [ ]:
fig,axs=plt.subplots(1,2,figsize=(12.5,4.6),layout="constrained")
for ax,mu in zip(axs,(0.0,0.15)):
    subset=[r for r in fit_rows if int(r["n"])==100 and int(r["d"])==2
            and float(r["mu"])==mu]
    assert subset, "Expected n=100, d=2 workloads for both mu values"
    for name,label,color in zip(names,pretty,colors):
        items=sorted((r for r in subset if r["method"]==name),
                     key=lambda r:int(r["M"]))
        ax.plot([int(r["M"]) for r in items],
                [1000*float(r["total_median_s"]) for r in items],
                marker="o", linestyle="-", lw=1.8,ms=4,label=label,color=color)
    ax.set(title=f"n=100, d=2, mu={mu:g}",
           xlabel="Penalty candidates M",ylabel="Total median milliseconds")
    ax.set_yscale("log")
    ax.grid(alpha=.2)
axs[0].legend(ncol=2,frameon=False,fontsize=8)
plt.show()


## Winner tables and explicit comparison against Cholesky

The winner depends on $n$, $d$, and the number $M$ of complete fits. For each fixed workload we take the minimum *measured* median, which is descriptive, **not an inferential significance claim**. Comparing both mean scenarios helps determine whether the implementation stays numerically correct and computationally stable when the difference-reference mean is nonzero.

In [ ]:
groups={}
for row in fit_rows:
    key=(int(row["n"]),int(row["d"]),int(row["M"]),float(row["mu"]))
    groups.setdefault(key,{})[row["method"]]=row
for key,items in sorted(groups.items()):
    if len(items)!=6:
        raise ValueError(f"Missing method for {key}")
    leader=min(names,key=lambda name:float(items[name]["total_median_s"]))
    chol=float(items["dense_cholesky"]["total_median_s"])
    bchol=float(items["banded_cholesky"]["total_median_s"])
    print(f"n={key[0]:3d} d={key[1]} M={key[2]:3d} mu={key[3]:.2f}  "
          f"winner={leader:17s}  dense-Chol/banded-Chol={chol/bchol:5.2f}x")


## Does $\mu\ne0$ change speed?

For each method, plot the ratio of measured times $T(\mu=0.15)/T(\mu=0)$ for matched $n,d,M$. The two fits have different right-hand sides but identical matrix bandwidth and factorization structure. A ratio near 1 is consistent with the computational model; departures may reflect measurement variability or RHS-specific arithmetic, **not** a change to the spectral decomposition, EDF, or theoretical complexity.

No pooled claim is made across machines. Data seeds are reused between $\mu$ values, but repeated timing blocks on one host are not independently randomized research replicates.

In [ ]:
keys=sorted({(int(r["n"]),int(r["d"]),int(r["M"]))
             for r in fit_rows})
fig,ax=plt.subplots(figsize=(12,4.2),layout="constrained")
for name,label,color in zip(names,pretty,colors):
    ratio=[]
    for n,d,M in keys:
        group=groups[(n,d,M,0.0)]
        other=groups[(n,d,M,0.15)]
        ratio.append(float(other[name]["total_median_s"])/
                     float(group[name]["total_median_s"]))
    ax.plot(range(len(keys)),ratio,marker="o", linestyle="-", lw=1.5,ms=3,
            color=color,label=label)
ax.axhline(1.0,color="#333f4f",ls="--",lw=1)
ax.set(ylabel="Measured time at mu=.15 / time at mu=0",
       xlabel="Matched (n, d, M) workloads",
       title="Paired mu comparison: same factorization, different RHS")
ax.set_xticks(range(len(keys)),[str(key) for key in keys],rotation=45,ha="right")
ax.legend(ncol=3,frameon=False,fontsize=8)
ax.grid(alpha=.2)
plt.show()


## EDF/trace-only computation (six methods)

For a fixed $\mu$, the hat matrix is

$$H_\lambda=(I+\lambda D_d^\top D_d)^{-1},\quad
\mathrm{EDF}(\lambda)=\mathrm{tr}(H_\lambda).$$

There is **no $\mu$ in this expression**, so doubling the trace experiment for different reference means would measure the same mathematical workload twice. Dense and banded Cholesky can compute traces by solving for $n$ right-hand sides at every penalty. The SVD sums inverse eigenvalue factors after preparing the decomposition.

In [ ]:
trace_workloads=sorted({(int(r["n"]),int(r["d"]),int(r["M"]))
                        for r in trace_rows})
fig,ax=plt.subplots(figsize=(11.5,4.4),layout="constrained")
for name,label,color in zip(names,pretty,colors):
    durations=[]
    for n,d,M in trace_workloads:
        row=next(r for r in trace_rows
                 if (int(r["n"]),int(r["d"]),int(r["M"]))==(n,d,M)
                 and r["method"]==name+"_trace")
        durations.append(1000*float(row["total_median_s"]))
    ax.plot(range(len(trace_workloads)),durations,marker="o",lw=1.8,
            ms=4,color=color,label=label)
ax.set_yscale("log")
ax.set_xticks(range(len(trace_workloads)),
              [f"({n}, {d}, {M})" for n,d,M in trace_workloads],rotation=20)
ax.set(xlabel="(n, d, M)",ylabel="Total median milliseconds",
       title="Trace/EDF profiles: six algorithms, no mu dependence")
ax.legend(ncol=3,frameon=False,fontsize=8)
ax.grid(alpha=.2)
plt.show()


## Interpretation and reproducibility

- **Dense Cholesky** exploits symmetry and positive definiteness but still allocates/factorizes an $n\times n$ dense matrix at every $\lambda$.
- **Banded Cholesky** also exploits the bandwidth $d$; it should be an especially relevant structural competitor for full fits, though actual speed depends on Python/SciPy overhead, matrix sizes, memory locality, hardware and BLAS.
- **Spectral SVD** has up-front decomposition cost and then cheap trace evaluations across many penalties.
- **General banded LU** (`solve_banded`) is not Cholesky; neither should be mislabeled.
- **Nonzero fixed $\mu$** changes fitted trends and RHS; it does not change the matrix, eigenvalues or trace EDF. Evaluating $\mu$ anew from data would be a different estimation problem.
- These are *solver* comparisons at supplied finite $\lambda$ values. They are not a global speed ranking for CV/GCV/AICc/BIC searches, nor a claim of causal superiority from one computer.

Windows 11 PowerShell reproduction:

```powershell
conda activate trendestimation
python -m pip install -e ".[docs,plot]" threadpoolctl
python paper\benchmarks\cholesky_comparison.py --preset full --repeats 7
# For local notebook rerender with your own measurements:
python paper\benchmarks\cholesky_comparison.py --preset full --repeats 7 --output-dir paper\benchmarks\cholesky_reference
python scripts\run_notebooks.py --only 12-six-solvers-cholesky-nonzero-mu --in-place --fail-fast
```

For publication-quality comparisons, use multiple independent seeds and sessions, report uncertainty of paired log runtime ratios and multiplicity-adjusted tests, and do not combine GitHub-hosted and Windows-VM timings in the same plot as if they came from one machine.